In [348]:
from sklearn import preprocessing
from sklearn.metrics import mean_absolute_percentage_error, r2_score, mean_absolute_percentage_error


# For 2D analysis
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit
from sklearn.preprocessing import MinMaxScaler
# from utils import period2freq, freq2period

# For PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.optimize import curve_fit

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import csv
import time
import glob
import os

MULTI_NODES = True # True: uses tasksPerNodes; False: uses parallelism
NORMALIZE = True
INITIAL_STAGE = False
DEBUG = False

# Datasize in KB
data_size_kb = {'4mb': 4096, '16mb': 16384, '64mb': 65536,
            '256mb': 262144, '512mb': 524288, '1gb': 1048576,
            '5gb': 5242880, '50gb': 52428800, '100gb': 104857600,
            '300gb': 314572800,}

# Key Parameters
WF_PARAMS = ['operation', 'randomOffset', 'transferSize', 
            'aggregateFilesizeMB', 'numTasks', 'parallelism', 'totalTime', 
            'numNodesList', 'numNodes', 'tasksPerNode', 'trMiB', 'storageType',
            'opCount','taskName','taskPID', 'fileName', 'stageOrder']

TARGET_PARAMS = [ "bestStorage" ]
op_dict = {0: "write", 1: "read"}

test_configs = {
    "1kg_10n_nfs": {
        "SCRIPT_ORDER": "1kg_script_order",
        "NUM_NODES_LIST": [10],
        "ALLOWED_PARALLELISM": [30],
        "exp_data_path": "./1kgenome/fastflow_tests", # ./1kgenome/fastflow_tests ./1kgenome/spm_tests
        "test_folders": ['par_6000_10n_nfs_ps300'] # par_6000_10n_nfs_ps300 par_6000_10n_pfs_ps300
    },
    "1kg_10n_0": {
        "SCRIPT_ORDER": "1kg_script_order",
        "NUM_NODES_LIST": [10],
        "ALLOWED_PARALLELISM": [30],
        "exp_data_path": "./1kgenome/spm_tests", # ./1kgenome/fastflow_tests ./1kgenome/spm_tests
        "test_folders": ['par_6000_10n_ssd_ps300_0'] # par_6000_10n_nfs_ps300 par_6000_10n_pfs_ps300
    },
    "1kg_10n_1": {
        "SCRIPT_ORDER": "1kg_script_order",
        "NUM_NODES_LIST": [10],
        "ALLOWED_PARALLELISM": [30],
        "exp_data_path": "./1kgenome/spm_tests", # ./1kgenome/fastflow_tests ./1kgenome/spm_tests
        "test_folders": ['par_6000_10n_ssd_ps300_1'] # par_6000_10n_nfs_ps300 par_6000_10n_pfs_ps300
    },
    "1kg_10n_2": {
        "SCRIPT_ORDER": "1kg_script_order",
        "NUM_NODES_LIST": [10],
        "ALLOWED_PARALLELISM": [30],
        "exp_data_path": "./1kgenome/spm_tests", # ./1kgenome/fastflow_tests ./1kgenome/spm_tests
        "test_folders": ['par_6000_10n_ssd_ps300_2'] # par_6000_10n_nfs_ps300 par_6000_10n_pfs_ps300
    },
    "1kg_10n_3": {
        "SCRIPT_ORDER": "1kg_script_order",
        "NUM_NODES_LIST": [10],
        "ALLOWED_PARALLELISM": [30],
        "exp_data_path": "./1kgenome/spm_tests", # ./1kgenome/fastflow_tests ./1kgenome/spm_tests
        "test_folders": ['par_6000_10n_ssd_ps300_3'] # par_6000_10n_nfs_ps300 par_6000_10n_pfs_ps300
    },
    "1kg_10n_pfs": {
        "SCRIPT_ORDER": "1kg_script_order",
        "NUM_NODES_LIST": [10],
        "ALLOWED_PARALLELISM": [30],
        "exp_data_path": "./1kgenome/spm_tests", # ./1kgenome/fastflow_tests ./1kgenome/spm_tests
        "test_folders": ['par_6000_10n_pfs_ps300'] # par_6000_10n_nfs_ps300 par_6000_10n_pfs_ps300
    },
     "ddmd_4n_l": { # normalize global
        "SCRIPT_ORDER": "ddmd_script_order",
        "NUM_NODES_LIST": [1, 2, 4 ],
        "exp_data_path": "./ddmd",
        "ALLOWED_PARALLELISM": [1, 3, 6, 12],
        "test_folders": ['ddmd_4n_pfs_large']
    },   
}

# Load experiment data
CURR_WF="ddmd_4n_l" # ddmd_2n_s, ddmd_4n_l, 1kg, pyflex_240f

SCRIPT_ORDER = test_configs[CURR_WF]["SCRIPT_ORDER"]
NUM_NODES_LIST = test_configs[CURR_WF]["NUM_NODES_LIST"]
ALLOWED_PARALLELISM = test_configs[CURR_WF]["ALLOWED_PARALLELISM"]
exp_data_path = test_configs[CURR_WF]["exp_data_path"]
test_folders = test_configs[CURR_WF]["test_folders"]

# test_folders = ['par_3000_1n_pfs_ps300', 'par_6000_1n_pfs_ps300', 
#                 'par_9000_1n_pfs_ps300'] par_3000_10n_shm_ps300

### DDMD
Total I/O time per taskName:
 - aggregate (write): 0.0305286564 (sec)
 - inference (write): 8.5899e-06 (sec)
 - openmm (write): 0.1809279096 (sec)
 - training (write): 0.586419552 (sec)
 - aggregate (read): 0.8245518099 (sec)
 - inference (read): 0.5552521026 (sec)
 - training (read): 3.4654465442999998 (sec)
Total I/O time per workflow: 5.6431351647

I/O Time Percentage (BeeGFS 2n)
- OpenMM: 0.17%
- Aggregate: 23.35%
- Training: 3.83%
- Inference: 3.70%
'''
OpenMM: 0.1809279096 / 106.5733333 = 
Aggregate: (0.8245518099 + 0.0305286564) / 3.662666667 =
Training: (3.4654465442999998 + 0.586419552) / 105.749	 =
Inference: (0.5552521026 + 8.5899e-06) / 14.992 =
'''

### 1000 Genome Test on SSD:

#### Trial 0
Total I/O time per taskName:
 - frequency (write): 0.011265510500000001 (sec)
 - individuals (write): 0.0027054705 (sec)
 - individuals_merge (write): 0.0008207719000000001 (sec)
 - mutation_overlap (write): 0.1407111768 (sec)
 - frequency (read): 0.45105542759999995 (sec)
 - individuals (read): 49.1197366278 (sec)
 - individuals_merge (read): 0.18885585 (sec)
 - mutation_overlap (read): 0.3870003042 (sec)
 - sifting (read): 0.2592469377 (sec)
Total I/O time per workflow: 50.56139807700001

Workflow Time
 - 'data stage-in for individuals : 0 minutes and 49 seconds elapsed (49 secs).'
 - 'individuals : 1 minutes and 2 seconds elapsed (62 secs).'
 - 'individuals_merge : 0 minutes and 23 seconds elapsed (23 secs).'
 - 'sifting : 0 minutes and 3 seconds elapsed (3 secs).'
 - 'mutation_overlap : 0 minutes and 18 seconds elapsed (18 secs).'
 - 'frequency : 6 minutes and 31 seconds elapsed (391 secs).'
 - 'All done : 9 minutes and 6 seconds elapsed (546 secs).'

"t0_task_write" : {
    "individuals": 0.0027054705, "individuals_merge": 0.0008207719000000001,
    "sifting": 0.0, "frequency": 0.011265510500000001, "mutation_overlap": 0.1407111768
},
"t0_task_read" : {
    "individuals": 49.1197366278, "individuals_merge": 0.18885585,
    "sifting": 0.2592469377, "frequency": 0.45105542759999995, "mutation_overlap": 0.3870003042
},
"t0_task_time" : {
    "individuals": 49, "individuals_merge": 62,
    "sifting": 3, "mutation_overlap": 18, "frequency": 391
}

#### Trial 1
Total I/O time per taskName:
 - frequency (write): 0.004702256 (sec)
 - individuals (write): 0.0028824532 (sec)
 - individuals_merge (write): 0.0008693151 (sec)
 - mutation_overlap (write): 0.09810824659999999 (sec)
 - frequency (read): 0.0333645987 (sec)
 - individuals (read): 56.766185590999996 (sec)
 - individuals_merge (read): 0.180459897 (sec)
 - mutation_overlap (read): 0.0377849895 (sec)
 - sifting (read): 0.2521806827 (sec)
Total I/O time per workflow: 57.376538029799995

Workflow Time
 - 'data stage-in for individuals : 1 minutes and 2 seconds elapsed (62 secs).'
 - 'individuals : 1 minutes and 17 seconds elapsed (77 secs).'
 - 'individuals_merge : 0 minutes and 21 seconds elapsed (21 secs).'
 - 'sifting : 0 minutes and 3 seconds elapsed (3 secs).'
 - 'mutation_overlap : 0 minutes and 15 seconds elapsed (15 secs).'
 - 'frequency : 5 minutes and 11 seconds elapsed (311 secs).'
 - 'All done : 8 minutes and 9 seconds elapsed (489 secs).'

t1_task_write : {
    "individuals": 0.0028824532, "individuals_merge": 0.0008693151,
    "sifting": 0.0, "frequency": 0.004702256, "mutation_overlap": 0.09810824659999999
},
t1_task_read : {
    "individuals": 56.766185590999996, "individuals_merge": 0.180459897,
    "sifting": 0.2521806827, "frequency": 0.0333645987, "mutation_overlap": 0.0377849895
},
t1_task_time : {
    "individuals": 62, "individuals_merge": 77,
    "sifting": 21, "mutation_overlap": 15, "frequency": 311
}

#### Trial 2
Total I/O time per taskName:
 - frequency (write): 0.0105912273 (sec)
 - individuals (write): 0.0031398628999999996 (sec)
 - individuals_merge (write): 0.0008452625999999999 (sec)
 - mutation_overlap (write): 0.028105798999999997 (sec)
 - frequency (read): 0.033347829 (sec)
 - individuals (read): 53.164005431999996 (sec)
 - individuals_merge (read): 0.178519392 (sec)
 - mutation_overlap (read): 0.037330755300000004 (sec)
 - sifting (read): 0.2547015924 (sec)
Total I/O time per workflow: 53.7105871525

Workflow Time
 - 'individuals : 1 minutes and 6 seconds elapsed (66 secs).'
 - 'individuals_merge : 0 minutes and 21 seconds elapsed (21 secs).'
 - 'sifting : 0 minutes and 3 seconds elapsed (3 secs).'
 - 'mutation_overlap : 0 minutes and 12 seconds elapsed (12 secs).'
 - 'frequency : 4 minutes and 5 seconds elapsed (245 secs).'
 - 'All done : 5 minutes and 56 seconds elapsed (356 secs).'

t2_task_write : {
    "individuals": 0.0031398628999999996, "individuals_merge": 0.0008452625999999999,
    "sifting": 0.0009596779999999999, "frequency": 0.0105912273, "mutation_overlap": 0.028105798999999997
},
t2_task_read : {
    "individuals": 53.164005431999996, "individuals_merge": 0.178519392,
    "sifting": 0.2547015924, "frequency": 0.033347829, "mutation_overlap": 0.037330755300000004
},
t2_task_time : {
    "individuals": 66, "individuals_merge": 21,
    "sifting": 3, "mutation_overlap": 12, "frequency": 245
}


#### Trial 3

Total I/O time per taskName:
 - individuals (write): 0.0034169434999999997 (sec)
 - individuals_merge (write): 0.000868328 (sec)
 - mutation_overlap (write): 0.12351022710000001 (sec)
 - frequency (read): 0.0153719148 (sec)
 - individuals (read): 53.79894497080001 (sec)
 - individuals_merge (read): 0.17899494600000002 (sec)
 - mutation_overlap (read): 0.0379941459 (sec)
 - sifting (read): 0.251910767 (sec)
Total I/O time per workflow: 54.41101224310001

Workflow Time
 - 'data stage-in for individuals : 1 minutes and 18 seconds elapsed (78 secs).'
 - 'individuals : 1 minutes and 17 seconds elapsed (77 secs).'
 - 'individuals_merge : 0 minutes and 22 seconds elapsed (22 secs).'
 - 'sifting : 0 minutes and 2 seconds elapsed (2 secs).'
 - 'mutation_overlap : 0 minutes and 16 seconds elapsed (16 secs).'
 - 'frequency : 1 minutes and 9 seconds elapsed (69 secs).'
 - 'All done : 4 minutes and 24 seconds elapsed (264 secs).'

"t3_task_write" : {
    "individuals": 0.0034169434999999997, "individuals_merge": 0.000868328,
    "sifting": 0.0009596779999999999, "frequency": 0.9, "mutation_overlap": 0.12351022710000001
},
"t3_task_read" : {
    "individuals": 53.79894497080001, "individuals_merge": 0.17899494600000002,
    "sifting": 0.251910767, "frequency": 0.0153719148, "mutation_overlap": 0.0379941459
},
"t3_task_time" : {
    "individuals": 78, "individuals_merge": 77,
    "sifting": 22, "mutation_overlap": 16, "frequency": 264
}


In [349]:
# My utility functions
import utils.perf_visualize as pv

# Parameter Notes for Datalife:
Each entry in the table represent only one single edge in the workflow. An directed edge connects a **fileName** and a **taskName**, representing data access.

---
- **operation**: The type of I/O operation {0: "write", 1: "read"}, value 1 represents read (e.g. a directed edge edge from a **fileName** to a **taskName**), value 0 represents write (e.g. a directed edge edge from a **taskName** to a **fileName**)
- **randomOffset**: The type of data access pattern { 0: "sequential file access", 1: "random file access"}
- **transferSize**: Average I/O size of the particular I/O operation to a file, calculated from aggregateFilesizeMB/opCount
- **aggregateFilesizeMB**: Total I/O size of a particular I/O operation to a file for a task
- **numTasks**: Number of parallel tasks for this particular task
- **totalTime**: The total I/O time of of a particular I/O operation to a file for a task
- **numNodes**: Number of nodes used for this particular task
- **tasksPerNode**: numTasks/numNodes for a task
- **bwMiB**: transferRate of a particular I/O operation to a file for a task, calculated from aggregateFilesizeMB/totalTime
- **storageType**: The storage type used in this task. {0: "localssd", 1: "beegfs/pfs", 2: "lustre", 3: "unknown"}
- **opCount**: the number of I/O operation count of a particular I/O operation to a file for a task
- **taskName**: the task name that is running for a particular workflow
- **taskPID**: the task PID
- **fileName**: the name of file that a I/O operation is for

In [350]:
def transform_store_code(storage_type):
    if storage_type == "localssd":
        store_code = 0
    elif storage_type == "beegfs" or storage_type == "pfs":
        store_code = 1
    elif storage_type == "lustre":
        store_code = 2
    else:
        store_code = 3
    return store_code

def decode_store_code(store_code):
    if store_code == 0:
        storage_type = "localssd"
    elif store_code == 1:
        storage_type = "beegfs"
    elif store_code == 2:
        storage_type = "lustre"
    else:
        storage_type = "unknown"
    return storage_type

def bytes_to_mb(file_size):
    """
    Convert a file size from bytes to megabytes (MB).
    
    Parameters:
    - file_size (str, int, or float): The file size in bytes (as an int/float) 
      or a string representation with size and unit (e.g., "1024 KiB").
      
    Returns:
    - float: The file size in MB.
    """
    # If file_size is a string, parse the value and unit
    if isinstance(file_size, str):
        size_num, size_unit = file_size.split()
        size_num = float(size_num)
        
        # Convert size to MB based on the unit
        if size_unit == "Bytes" or size_unit == "B":
            return size_num / (1024 ** 2)  # Convert bytes to MB
        elif size_unit == "KiB":
            return size_num / 1024  # Convert KiB to MB
        elif size_unit == "MiB":
            return size_num  # Already in MB
        elif size_unit == "GiB":
            return size_num * 1024  # Convert GiB to MB
        elif size_unit == "TiB":
            return size_num * (1024 ** 2)  # Convert TiB to MB
        else:
            raise ValueError(f"Unknown size unit: {size_unit}")
    elif isinstance(file_size, (int, float)):
        # If file_size is an integer or float, assume it's in bytes
        return file_size / (1024 ** 2)  # Convert bytes to MB
    else:
        raise TypeError("file_size must be a string or a number")


In [351]:
def is_sequential(numbers):
    if not numbers:  # Check if the list is empty
        return False

    sorted_numbers = sorted(numbers)  # Sort the numbers
    return all(sorted_numbers[i] + 1 == sorted_numbers[i + 1] for i in range(len(sorted_numbers) - 1))


def get_stat_file_pids(all_files):
    # Extract target tasks from blk_files
    target_tasks = set()
    for blk_file in all_files:
        # Get the filename without the path
        filename = os.path.basename(blk_file)
        # repalce ".local" for now
        filename = filename.replace(".local", "")
        
        # Split filename by '.'
        parts = filename.split('.')
        # print(f"get_stat_file_pids() : parts = {parts}")
        if len(parts) >= 3:
            # Get the target task from the -3 extension
            task = parts[-3]
            target_tasks.add(task)
    target_tasks = sorted(target_tasks)
    return target_tasks

import os
import glob
import json

import os
import json

def add_stat_to_df(trial_folder, monitor_timer_stat_io, 
                   operation, fname, task_pid, store_code):
    # Process the file name
    fname = fname.replace(".local", ".")
    fileName = ".".join(fname.split(".")[:-4])  # Remove last 4 extensions
    fileName = os.path.basename(fileName)      # Keep only the basename
    
    # Initialize statistics
    tmp_write_stat = {
        'aggregateFilesizeMB': bytes_to_mb(monitor_timer_stat_io[2]),
        'transferSize': monitor_timer_stat_io[2] / monitor_timer_stat_io[1],
        'operation': int(operation),
        'totalTime': monitor_timer_stat_io[0],
        'trMiB': bytes_to_mb(monitor_timer_stat_io[2] / monitor_timer_stat_io[0]),
        'storageType': store_code,
        'opCount': monitor_timer_stat_io[1],
        'taskPID': task_pid,
        'fileName': fileName,
    }
    print(f"fileName = {fileName}")
    
    if tmp_write_stat['totalTime'] > 100:
        print(f"Recorded large totalTime[{monitor_timer_stat_io}] from task_pid[{task_pid}] fileName[{fileName}]")
    
    if "6818-dc111" in task_pid:
        print(f"monitor_timer_stat_io: {monitor_timer_stat_io}")
    
    # Determine operation type
    op = "w" if operation == 0 else "r"
    
    # Ensure the trial folder exists
    if not os.path.exists(trial_folder):
        print(f"Trial folder does not exist: {trial_folder}")
        return tmp_write_stat
    
    # List all files in the trial folder
    all_files = os.listdir(trial_folder)
    # print(f"Total files in {trial_folder}: {len(all_files)}")
    
    # Find matching files using substring matching
    matching_files = [
        os.path.join(trial_folder, file)
        for file in all_files
        if f"{fileName}.{task_pid}.local.{op}" in file
    ]
    
    # if len(matching_files) == 0:
    #     print(f"No matching files found for fileName[{fileName}] task_pid[{task_pid}] op[{op}]")
    # else:
    #     print(f"Found {len(matching_files)} matching files: {matching_files}")
    
    # Process matching files to determine write pattern
    write_pattern = 0  # 0: seq, 1: rand
    for matching_file in matching_files:
        try:
            with open(matching_file) as f:
                w_blk_trace_data = json.load(f)
                blk_list = w_blk_trace_data.get('io_blk_range', [])
                
                # Validate blk_list length
                if len(blk_list) >= 4:
                    if blk_list[3] == -2:
                        write_pattern = 1
                        print(f"Detected random write pattern in file: {matching_file}")
                        break
                else:
                    print(f"Warning: Invalid 'io_blk_range' in file: {matching_file}")
        except Exception as e:
            print(f"Error processing file {matching_file}: {e}")
    
    # Update statistics
    tmp_write_stat['randomOffset'] = write_pattern
    
    return tmp_write_stat

    

# TODO: find task PID's input and output to match script name
def get_wf_result_df(tests, WF_PARAMS, target_tasks, storageType="localssd"):
    wf_df = pd.DataFrame(columns=WF_PARAMS)

    # Identify trial folders
    wf_trial_folders = [
        folder for folder in glob.glob(f"{tests}/*")
        if folder.endswith(("t1", "t2", "t3"))
    ]
    print(f"Trial folders: {wf_trial_folders}")

    store_code = transform_store_code(storageType)

    for trial_folder in wf_trial_folders:
        blk_files = glob.glob(f"{trial_folder}/*_blk_trace.json")
        datalife_jsons = glob.glob(f"{trial_folder}/*.datalife.json")
        target_tasks = get_stat_file_pids(blk_files)

        for datalife_json in datalife_jsons:
            task_pid = os.path.basename(datalife_json).split(".")[1]
            if task_pid not in target_tasks:
                continue

            try:
                with open(datalife_json) as f:
                    datalife_data = json.load(f)
            except json.JSONDecodeError:
                print(f"Error loading file: {datalife_json}")
                continue

            task_name = list(datalife_data.keys())[0]
            monitor_timer_stat = datalife_data[task_name]['monitor']
            system_timer_stat = datalife_data[task_name]['system']
            monitor_timer_targets = ["read", "write"]

            for fname in [f for f in blk_files if f".{task_pid}." in f]:
                op_type = "read" if ".r_blk_trace." in fname else "write"
                monitor_stat = monitor_timer_stat[op_type]

                if bytes_to_mb(monitor_stat[2]) == 0:
                    print(f"No {op_type} stat for task_name[{task_name}] task_pid[{task_pid}]")
                    continue
                
                # taskParallelism = task_name_to_parallelism[task_name]
                tmp_stat = add_stat_to_df(
                    trial_folder, monitor_stat, 
                    1 if op_type == "read" else 0,
                    fname, task_pid, store_code
                )
                wf_df = wf_df._append(tmp_stat, ignore_index=True)

    return wf_df

In [352]:
target_tasks = ["python"] # omit srun from 1kgenome run

all_wf_df = pd.DataFrame(columns=WF_PARAMS)

def get_test_folder_dfs(test_folder, WF_PARAMS, target_tasks,
                        storageType="localssd", workflow="1kg"):
    folder_dfs = pd.DataFrame(columns=WF_PARAMS)

    for tests in test_folder:
        stat_path = f"{exp_data_path}/{tests}"

        # Generate workflow data
        wf_df = get_wf_result_df(stat_path, WF_PARAMS, target_tasks,
                                 storageType=storageType)
        print(wf_df.head(5))
        print(f"df shape: {wf_df.shape}")

        # Append workflow data to the folder dataframe
        folder_dfs = folder_dfs._append(wf_df, ignore_index=True)
        
    return folder_dfs

wf_pfs_df = pd.DataFrame(columns=WF_PARAMS)



wf_pfs_df = wf_pfs_df._append(get_test_folder_dfs(test_folders, 
                                        WF_PARAMS, target_tasks,
                                        storageType="pfs", workflow="ddmd"), ignore_index=True)



Trial folders: ['./ddmd/ddmd_4n_pfs_large/4n_pfs_t1']
fileName = stage0000_task0000.h5
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_task0000.h5.190075-dlt02.local.w_blk_trace.json
fileName = stage0000_task0000.dcd
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_task0000.dcd.190075-dlt02.local.w_blk_trace.json
fileName = stage0000_task0007.h5
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_task0007.h5.190758-dlt02.local.r_blk_trace.json
fileName = stage0000_task0002.h5
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_task0002.h5.190758-dlt02.local.r_blk_trace.json
fileName = stage0000_task0010.h5
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_task0010.h5.190758-dlt02.local.r_blk_trace.json
fileName = stage0000_task0000.h5
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_ta

/tmp/ipykernel_123350/2549204596.py:161: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_df = wf_df._append(tmp_stat, ignore_index=True)


fileName = stage0000_task0005.dcd
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_task0005.dcd.170276-dlt04.local.w_blk_trace.json
fileName = stage0000_task0005.h5
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_task0005.h5.170276-dlt04.local.w_blk_trace.json
fileName = stage0000_task0009.dcd
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_task0009.dcd.74813-dlt06.local.w_blk_trace.json
fileName = stage0000_task0009.h5
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_task0009.h5.74813-dlt06.local.w_blk_trace.json
fileName = stage0000_task0003.dcd
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_task0003.dcd.170277-dlt04.local.w_blk_trace.json
fileName = stage0000_task0003.h5
Detected random write pattern in file: ./ddmd/ddmd_4n_pfs_large/4n_pfs_t1/stage0000_task0003.h5.170277-dlt04.local.w_blk_trace.json
fileNa

/tmp/ipykernel_123350/2648307957.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  folder_dfs = folder_dfs._append(wf_df, ignore_index=True)
/tmp/ipykernel_123350/2648307957.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_pfs_df = wf_pfs_df._append(get_test_folder_dfs(test_folders,


In [353]:
wf_pfs_df.shape, wf_pfs_df.columns

((89, 17),
 Index(['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMB',
        'numTasks', 'parallelism', 'totalTime', 'numNodesList', 'numNodes',
        'tasksPerNode', 'trMiB', 'storageType', 'opCount', 'taskName',
        'taskPID', 'fileName', 'stageOrder'],
       dtype='object'))

In [354]:
def match_script_name(tests):
    # Find folders ending with [t1, t2, t3] in the test_folders
    test_folders = glob.glob(f"{tests}/*")
    wf_trial_folders = [folder for folder in test_folders if folder.endswith("t1") or folder.endswith("t2") or folder.endswith("t3")]
    print(f"Trial folders: {wf_trial_folders}")

    pid_input_output_dict = {}

    for trial_folder in wf_trial_folders:
        blk_files = glob.glob(f"{trial_folder}/*_blk_trace.json")
        print(f"len(blk_files) = {len(blk_files)}")
        unique_pids = get_stat_file_pids(blk_files)

        for pid in unique_pids:
            if pid not in pid_input_output_dict:
                pid_input_output_dict[pid] = {
                    "input": [],
                    "output": [],
                    "prevTask": "",
                    "taskName": ""
                }

            # Find the blk_trace_jsons files with the current task_pid
            w_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{pid}.local.w_blk_trace.json")
            r_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{pid}.local.r_blk_trace.json")

            # Replace ".local" with an empty string
            w_blk_trace_jsons = [f.replace(".local", "") for f in w_blk_trace_jsons]
            r_blk_trace_jsons = [f.replace(".local", "") for f in r_blk_trace_jsons]

            # Process write (output) files
            for w_file_path in w_blk_trace_jsons:
                w_file_name_parts = w_file_path.split(".")
                w_file_name = '.'.join(w_file_name_parts[:-3])  # Remove the last 3 extensions
                w_file_basename = os.path.basename(w_file_name)
                pid_input_output_dict[pid]['output'].append(w_file_basename)

            # Process read (input) files
            for r_file_path in r_blk_trace_jsons:
                r_file_name_parts = r_file_path.split(".")
                r_file_name = '.'.join(r_file_name_parts[:-3])  # Remove the last 3 extensions
                r_file_basename = os.path.basename(r_file_name)
                if r_file_basename not in pid_input_output_dict[pid]['input']:
                    pid_input_output_dict[pid]['input'].append(r_file_basename)

    return pid_input_output_dict

def get_wf_pid_script_dict(test_folder):

    all_wf_dict= {}

    for tests in test_folder:
        # # check of test folder starts with seq or par
        # if tests.startswith("seq"):
        #     numTasksWrite = 1
        #     numTasksRead = 1
        # else:
        #     numTasksWrite = 1
        #     numTasksRead = 1

        # io_size_dfs
        wf_dict = match_script_name(f"{exp_data_path}/{tests}")

        # # corr_matrix(wf_df, storageType)
        all_wf_dict.update(wf_dict)
    return all_wf_dict


all_wf_dict = get_wf_pid_script_dict(test_folders)

print(all_wf_dict)



Trial folders: ['./ddmd/ddmd_4n_pfs_large/4n_pfs_t1']
len(blk_files) = 89
{'143674-dlt05': {'input': [], 'output': ['stage0000_task0007.dcd', 'stage0000_task0007.h5'], 'prevTask': '', 'taskName': ''}, '143675-dlt05': {'input': [], 'output': ['stage0000_task0008.dcd', 'stage0000_task0008.h5'], 'prevTask': '', 'taskName': ''}, '143676-dlt05': {'input': [], 'output': ['stage0000_task0006.dcd', 'stage0000_task0006.h5'], 'prevTask': '', 'taskName': ''}, '170276-dlt04': {'input': [], 'output': ['stage0000_task0005.dcd', 'stage0000_task0005.h5'], 'prevTask': '', 'taskName': ''}, '170277-dlt04': {'input': [], 'output': ['stage0000_task0003.dcd', 'stage0000_task0003.h5'], 'prevTask': '', 'taskName': ''}, '170278-dlt04': {'input': [], 'output': ['stage0000_task0004.dcd', 'stage0000_task0004.h5'], 'prevTask': '', 'taskName': ''}, '190075-dlt02': {'input': [], 'output': ['stage0000_task0000.h5', 'stage0000_task0000.dcd'], 'prevTask': '', 'taskName': ''}, '190099-dlt02': {'input': [], 'output': ['s

In [355]:
# Add prevTask column
wf_pfs_df['prevTask'] = ""
wf_pfs_df['taskName'] = "unknown"
print(wf_pfs_df.head(5))
print(wf_pfs_df.shape)

  operation randomOffset  transferSize  aggregateFilesizeMB numTasks  \
0         0            1   1066.397658             7.293896      NaN   
1         0            1   1066.397658             7.293896      NaN   
2         1            1    354.246967            17.099243      NaN   
3         1            1    354.246967            17.099243      NaN   
4         1            1    354.246967            17.099243      NaN   

  parallelism  totalTime numNodesList numNodes tasksPerNode       trMiB  \
0         NaN   0.026276          NaN      NaN          NaN  277.589578   
1         NaN   0.026276          NaN      NaN          NaN  277.589578   
2         NaN   0.211424          NaN      NaN          NaN   80.876723   
3         NaN   0.211424          NaN      NaN          NaN   80.876723   
4         NaN   0.211424          NaN      NaN          NaN   80.876723   

  storageType opCount taskName       taskPID                fileName  \
0           1    7172  unknown  190075-dlt02

In [356]:
# save to initial df
wf_pfs_df.to_csv(f'first_df.csv', index=False)

In [357]:
import re
        
def matches_pattern(file_path, patterns):
    """Match a file path against task definition patterns."""
    file_name = os.path.basename(file_path)
    for pattern in patterns:
        try:
            regex_pattern = re.compile(pattern)
            if regex_pattern.fullmatch(file_name):
                return True
        except re.error as e:
            print(f"Invalid regex: {pattern}, Error: {e}")
    return False

def assign_task_names(tasks, task_order_dict):
    """Assign task names and predecessors to tasks based on patterns."""
    for task_pid, details in tasks.items():
        input_paths = details.get('input', [])
        output_paths = details.get('output', [])
        task_name = details.get('taskName', 'unknown')  # Use existing or default to 'unknown'

        # Iterate through each task definition
        for task, definition in task_order_dict.items():
            # Check if any output matches
            if any(matches_pattern(op, definition['outputs']) for op in output_paths):
                task_name = task
                tasks[task_pid]['taskName'] = task_name
                tasks[task_pid]['stage_order'] = definition['stage_order']
                break

            # If no output matches, check for input matches
            for prevTask, predecessor_def in definition['predecessors'].items():
                if any(matches_pattern(ip, predecessor_def.get('inputs', [])) for ip in input_paths):
                    task_name = task
                    tasks[task_pid]['taskName'] = task_name
                    tasks[task_pid]['stage_order'] = definition['stage_order']
                    tasks[task_pid]['prevTask'] = prevTask
                    # print(f"Input match found: Task [{task}] prevTask [{prevTask}] with input_patterns {predecessor_def.get('inputs', [])}")
                    break

        # If no valid match, warn about the task
        if task_name == 'unknown':
            print(f"Warning: Task PID {task_pid} could not be assigned a valid taskName.")

    return tasks
        
# Load task ordering json file
task_order_dict = {}
with open(f"{exp_data_path}/{SCRIPT_ORDER}.json") as f:
    task_order_dict = json.load(f)

print(f"task_order_dict : {task_order_dict}")
# Create a mapping from taskName to parallelism
task_name_to_parallelism = {task: info['parallelism'] for task, info in task_order_dict.items()}
# print(task_name_to_parallelism)

# Fill in task names
assign_task_names(all_wf_dict, task_order_dict)
# Unique list of taskNames
taskNames = set([v['taskName'] for v in all_wf_dict.values()])
print(f"Unique taskNames: {taskNames}")
print(f"all_wf_dict:")
for k,v in all_wf_dict.items():
    print(f"{k}:{v}")
print(f"all_wf_dict-----")


if DEBUG:
    print(wf_pfs_df['fileName'].unique())
    print(wf_pfs_df['taskName'].unique())

task_order_dict : {'openmm': {'stage_order': 0, 'parallelism': 12, 'num_tasks': 12, 'predecessors': {'initial_data': {'inputs': []}}, 'outputs': ['stage\\d{4}_task\\d{4}\\.dcd', 'stage\\d{4}_task\\d{4}\\.h5']}, 'aggregate': {'stage_order': 1, 'parallelism': 1, 'num_tasks': 1, 'predecessors': {'openmm': {'inputs': ['stage\\d{4}_task\\d{4}\\.h5']}}, 'outputs': ['aggregated.h5']}, 'training': {'stage_order': 1, 'parallelism': 1, 'num_tasks': 1, 'predecessors': {'openmm': {'inputs': ['stage\\d{4}_task\\d{4}\\.h5']}, 'aggregate': {'inputs': ['aggregated.h5']}}, 'outputs': ['virtual_stage0000+_task[0-9]+\\.h5', 'embeddings-epoch-[0-9]+-[0-9]{8}-[0-9]{6}\\.h5', 'epoch-[0-9]+-[0-9]{8}-[0-9]{6}\\.pt', 'generator-weights\\.pt', 'encoder-weights\\.pt', 'discriminator-weights\\.pt']}, 'inference': {'stage_order': 1, 'parallelism': 1, 'num_tasks': 1, 'predecessors': {'openmm': {'inputs': ['stage\\d{4}_task\\d{4}\\.h5']}}, 'outputs': ['virtual_stage0003+_task[0-9]+\\.h5']}}
Unique taskNames: {'openm

In [358]:
# Create a mapping from taskPID to taskName
task_pid_to_name = {pid: info['taskName'] for pid, info in all_wf_dict.items()}
# print(task_pid_to_name)
# Update the DataFrame with the taskName
wf_pfs_df['taskName'] = wf_pfs_df['taskPID'].map(task_pid_to_name).fillna('unknown')

# Create a mapping from taskPID to prevTask
task_pid_to_prod_task = {pid: info['prevTask'] for pid, info in all_wf_dict.items()}
# print(task_pid_to_prod_task)
# add prevTask column to the DataFrame
wf_pfs_df['prevTask'] = wf_pfs_df['taskPID'].map(task_pid_to_prod_task).fillna('unknown')

# Print the updated DataFrame
# print(wf_pfs_df.head(5))
print(wf_pfs_df.shape)
df_unknown = wf_pfs_df[wf_pfs_df['taskName'] == 'unknown']
print(f"df unknown ({df_unknown.shape}):\n{df_unknown.head(5)}")
# print(f"df found:\n{wf_pfs_df[wf_pfs_df['taskName'] != 'unknown']}")

for pid, info in all_wf_dict.items():
    if 'stage_order' not in info:
        print(f"Missing 'stage_order' for taskPID: {pid}, info: {info}")

# Create a mapping from taskPID to stage_order
task_pid_to_stage_order = {pid: info['stage_order'] for pid, info in all_wf_dict.items()}
# print(task_pid_to_stage_order)
# add prevTask column to the DataFrame
wf_pfs_df['stageOrder'] = wf_pfs_df['taskPID'].map(task_pid_to_stage_order).fillna('-1')


# remove rows with filename contaiing string "SIFT.chr*.vcf"
for chrom in range(0, 11):
    wf_pfs_df = wf_pfs_df[~wf_pfs_df['fileName'].str.contains(f"SIFT.chr{chrom}.vcf")]
    
# Adjust dataframe prevTask
for index, row in wf_pfs_df.iterrows():
    if row['operation'] == 0:
        if row['taskName'] == '':
            # Update taskName for write tasks to "" (empty string)
            wf_pfs_df.at[index, 'taskName'] = 'none'
    else:
        # Adjust read task predecessors
        taskName = row['taskName']
        fileName = row['fileName']
        task_definition = task_order_dict[taskName]
        for task, inputs in task_definition['predecessors'].items():
            input_patterns = inputs['inputs']
            if matches_pattern(fileName, input_patterns):
                wf_pfs_df.at[index, 'prevTask'] = task



(89, 18)
df unknown ((0, 18)):
Empty DataFrame
Columns: [operation, randomOffset, transferSize, aggregateFilesizeMB, numTasks, parallelism, totalTime, numNodesList, numNodes, tasksPerNode, trMiB, storageType, opCount, taskName, taskPID, fileName, stageOrder, prevTask]
Index: []


In [359]:
# print(f"df found:\n{wf_pfs_df[wf_pfs_df['taskName'] == 'trackstats']}")


# # Below values can only be updated once task name and per task parallelism is known
import math

# Assuming 'wf_pfs_df' is the DataFrame with a 'taskName' column
for index, row in wf_pfs_df.iterrows():
    task_name = row['taskName']
    if task_name in task_name_to_parallelism:
        task_parallelism = task_name_to_parallelism[task_name]

        
        row['numNodesList'] = NUM_NODES_LIST
        # Update numTasks and tasksPerNode
        row['numTasks'] = task_parallelism
        # row['tasksPerNode'] = math.ceil(task_parallelism / NUM_NODES_LIST)

        # Update the DataFrame
        wf_pfs_df.at[index, 'numNodesList'] = row['numNodesList']
        wf_pfs_df.at[index, 'numTasks'] = task_parallelism
        # wf_pfs_df.at[index, 'tasksPerNode'] = math.ceil(task_parallelism / NUM_NODES_LIST)
        wf_pfs_df.at[index, 'parallelism'] = task_parallelism

if DEBUG:
    print(wf_pfs_df.shape)
    print(wf_pfs_df.head())


In [360]:
# Expand the dataframe for multi-nodes configuration calculation
def expand_df(wf_pfs_df):

    # Create a new DataFrame to store updated rows
    updated_rows = []

    # Iterate through each row in the DataFrame
    for index, row in wf_pfs_df.iterrows():
        num_nodes_list = row['numNodesList']  # Extract the list of numNodes
        
        for num_nodes in num_nodes_list:
            # Create a copy of the current row
            new_row = row.copy()
            
            # Update the numNodes and tasksPerNode for the new row
            tasksPerNode = math.ceil(row['parallelism'] / num_nodes)
            new_row['tasksPerNode'] = tasksPerNode
            new_row['numNodes'] = num_nodes
            
            
            # Append the updated row to the list
            updated_rows.append(new_row)

    # Create a new DataFrame with the updated rows
    expanded_df = pd.DataFrame(updated_rows)

    # Reset the index of the expanded DataFrame
    expanded_df.reset_index(drop=True, inplace=True)

    # # Print the updated DataFrame for verification
    # print(expanded_df.shape)
    # print(expanded_df.head())
    
    return expanded_df


if MULTI_NODES:
    wf_pfs_df = expand_df(wf_pfs_df)
    # Print the updated DataFrame for verification
    if DEBUG:
        print(wf_pfs_df.shape)
        print(wf_pfs_df.head())

# for rows when parallelism is 1, update numNodes to 1
for index, row in wf_pfs_df.iterrows():
    if row['parallelism'] == 1:
        wf_pfs_df.at[index, 'numNodes'] = 1

In [361]:
# # Modify numTasks by mapping the "parallelism" from task_order_dict based on taskName

# wf_pfs_df['numTasks'] = wf_pfs_df['taskName'].map(task_name_to_parallelism).fillna(1)



In [362]:
# Save the updated DataFrame to a CSV file
wf_pfs_df.to_csv(f'{test_folders[0]}.csv', index=False)

task_name_to_parallelism

{'openmm': 12, 'aggregate': 1, 'training': 1, 'inference': 1}

In [363]:
# Calculate I/O time per taskName
write_sub_df = wf_pfs_df[wf_pfs_df['operation'] == 0]
read_sub_df = wf_pfs_df[wf_pfs_df['operation'] == 1]

task_io_time_total = wf_pfs_df.groupby('taskName')['totalTime'].sum()
task_io_time_write = write_sub_df.groupby('taskName')['totalTime'].sum()
task_io_time_read = read_sub_df.groupby('taskName')['totalTime'].sum()

task_io_time_adjust = {"read": 0, "write": 0}
total_wf_io_time = 0
total_wf_io_time_write = 0
total_wf_io_time_read = 0
print("Total I/O time per taskName:")
for task, write_time in task_io_time_write.items():
    # Adjust I/O time by parallelism
    write_time_adjusted = write_time /10 #/ (task_name_to_parallelism[task] * len(NUM_NODES_LIST))
    task_io_time_adjust["write"]+=write_time_adjusted
    total_wf_io_time_write+=write_time_adjusted
    total_wf_io_time+=write_time_adjusted
    print(f" - {task} (write): {write_time_adjusted} (sec)")
for task, read_time in task_io_time_read.items():
    # Adjust I/O time by parallelism
    read_time_adjusted = read_time /10 #/ (task_name_to_parallelism[task] * len(NUM_NODES_LIST))
    task_io_time_adjust["read"]+=read_time_adjusted
    total_wf_io_time_read+=read_time_adjusted
    total_wf_io_time+=read_time_adjusted
    print(f" - {task} (read): {read_time_adjusted} (sec)")
    
print(f"Total I/O time per workflow: {total_wf_io_time}")


# print("Total I/O time per stage:")
# for task, io_time in task_io_time_total.items():
#     # Adjust I/O time by parallelism
#     io_time_adjusted = io_time / task_name_to_parallelism[task]
#     task_io_time_adjust[task] = io_time_adjusted
#     total_wf_io_time+=io_time_adjusted
    
#     print(f" {task}: {io_time_adjusted} (sec)")

# # print(task_io_time_adjust)
# print(f"Total I/O time per workflow: {total_wf_io_time}")

Total I/O time per taskName:
 - aggregate (write): 0.0305286564 (sec)
 - inference (write): 8.5899e-06 (sec)
 - openmm (write): 0.1809279096 (sec)
 - training (write): 0.586419552 (sec)
 - aggregate (read): 0.8245518099 (sec)
 - inference (read): 0.5552521026 (sec)
 - training (read): 3.4654465442999998 (sec)
Total I/O time per workflow: 5.6431351647


In [364]:
# Read from file "./master_ior_df.csv"
df_ior = pd.read_csv("./master_ior_df.csv")
# df_ior = pd.read_csv(f'{test_folders[0]}.csv')
print(df_ior.columns)
print(df_ior.shape)

# oscache size is 25GiB
oscacheSizeMB = 25 * 1024  # Convert to MiB

Index(['Unnamed: 0', 'operation', 'randomOffset', 'transferSize',
       'aggregateFilesizeMB', 'numTasks', 'totalTime_ssd', 'numNodes',
       'tasksPerNode', 'parallelism', 'trMiB_ssd', 'trMiB_ave_ssd',
       'trMiB_ave_std_ssd', 'trMiB_ave_std_perc_ssd',
       'aggregateFilesizeMB_log_ssd', 'transferSize_log_ssd',
       'aggregateFilesizeMB_log_norm_ssd', 'trMiB_norm_ssd',
       'operation_beegfs', 'totalTime_beegfs', 'trMiB_beegfs',
       'trMiB_ave_beegfs', 'trMiB_ave_std_beegfs', 'trMiB_ave_std_perc_beegfs',
       'aggregateFilesizeMB_log_beegfs', 'transferSize_log_beegfs',
       'aggregateFilesizeMB_log_norm_beegfs', 'trMiB_norm_beegfs',
       'selectStorage'],
      dtype='object')
(24900, 29)


In [365]:
t0_dict = {
    "t0_task_write" : {
    "individuals": 0.0027054705, "individuals_merge": 0.0008207719000000001,
    "sifting": 0.0, "frequency": 0.011265510500000001, "mutation_overlap": 0.1407111768
    },
    "t0_task_read" : {
        "individuals": 49.1197366278, "individuals_merge": 0.18885585,
        "sifting": 0.2592469377, "frequency": 0.45105542759999995, "mutation_overlap": 0.3870003042
    },
    "t0_task_time" : {
        "individuals": 49, "individuals_merge": 62,
        "sifting": 3, "mutation_overlap": 18, "frequency": 391
    }
}

t3_dict = {
    "t3_task_write" : {
        "individuals": 0.0034169434999999997, "individuals_merge": 0.000868328,
        "sifting": 0.0009596779999999999, "frequency": 0.9, "mutation_overlap": 0.12351022710000001
    },
    "t3_task_read" : {
        "individuals": 53.79894497080001, "individuals_merge": 0.17899494600000002,
        "sifting": 0.251910767, "frequency": 0.0153719148, "mutation_overlap": 0.0379941459
    },
    "t3_task_time" : {
        "individuals": 78, "individuals_merge": 77,
        "sifting": 22, "mutation_overlap": 16, "frequency": 264
    }
}

t2_dict = {
    "t2_task_write" : {
        "individuals": 0.0031398628999999996, "individuals_merge": 0.0008452625999999999,
        "sifting": 0.0009596779999999999, "frequency": 0.0105912273, "mutation_overlap": 0.028105798999999997
    },
    "t2_task_read" : {
        "individuals": 53.164005431999996, "individuals_merge": 0.178519392,
        "sifting": 0.2547015924, "frequency": 0.033347829, "mutation_overlap": 0.037330755300000004
    },
    "t2_task_time" : {
        "individuals": 66, "individuals_merge": 21,
        "sifting": 3, "mutation_overlap": 12, "frequency": 245
    }
}

t1_dict = {
    "t1_task_write" : {
        "individuals": 0.0028824532, "individuals_merge": 0.0008693151,
        "sifting": 0.0, "frequency": 0.004702256, "mutation_overlap": 0.09810824659999999
    },
    "t1_task_read" : {
        "individuals": 56.766185590999996, "individuals_merge": 0.180459897,
        "sifting": 0.2521806827, "frequency": 0.0333645987, "mutation_overlap": 0.0377849895
    },
    "t1_task_time" : {
        "individuals": 62, "individuals_merge": 77,
        "sifting": 21, "mutation_overlap": 15, "frequency": 311
    }
}





In [366]:
import numpy as np

# Function to calculate computation time, I/O time, and percentages for each task
def calculate_metrics(data_dict):
    computation_time = {}
    io_time = {}
    io_time_percentage = {}
    task_time_percentages = {}
    read_time_percentage = {}
    write_time_percentage = {}

    # Extract keys dynamically
    write_key = [key for key in data_dict if "task_write" in key][0]
    read_key = [key for key in data_dict if "task_read" in key][0]
    time_key = [key for key in data_dict if "task_time" in key][0]

    # Calculate the complete workflow time (sum of all task times)
    workflow_time = sum(data_dict[time_key].values())

    # Loop through tasks
    for task in data_dict[time_key]:
        # Get task times
        task_time = data_dict[time_key][task]
        write_time = data_dict[write_key][task]
        read_time = data_dict[read_key][task]

        # Calculate computation time, I/O time, and I/O percentage
        comp_time = task_time - (write_time + read_time)
        task_io_time = write_time + read_time
        io_percentage = (task_io_time / task_time) * 100 if task_time != 0 else 0

        # Calculate percentage of this task's time over the workflow time
        task_time_percentage = (task_time / workflow_time) * 100 if workflow_time != 0 else 0

        # Calculate read time percentage over I/O time
        read_percentage = (read_time / task_io_time) * 100 if task_io_time != 0 else 0

        # Calculate write time percentage over I/O time
        write_percentage = (write_time / task_io_time) * 100 if task_io_time != 0 else 0

        # Store results for each task
        computation_time[task] = comp_time
        io_time[task] = task_io_time
        io_time_percentage[task] = io_percentage
        task_time_percentages[task] = task_time_percentage
        read_time_percentage[task] = read_percentage
        write_time_percentage[task] = write_percentage

    return computation_time, io_time, io_time_percentage, task_time_percentages, read_time_percentage, write_time_percentage

# Function to calculate deviation percentage
def calculate_deviation_percentage(times_across_trials):
    deviation_percentages = {}
    for task in times_across_trials[0].keys():  # Assuming all trials have the same tasks
        # Collect times for the task across trials
        task_times = [trial[task] for trial in times_across_trials]

        # Calculate mean and standard deviation for the task
        mean_time = np.mean(task_times)
        std_time = np.std(task_times)

        # Calculate deviation percentage
        deviation_percentage = (std_time / mean_time) * 100 if mean_time != 0 else 0
        deviation_percentages[task] = deviation_percentage

    return deviation_percentages

# Calculate metrics for each trial
computation_times = []
io_times = []
io_percentages = []
task_time_percentages = []
read_time_percentages = []
write_time_percentages = []

for trial in [t1_dict, t2_dict, t3_dict]:
    comp_time, io_time, io_percentage, task_time_percentage, read_percentage, write_percentage = calculate_metrics(trial)
    computation_times.append(comp_time)
    io_times.append(io_time)
    io_percentages.append(io_percentage)
    task_time_percentages.append(task_time_percentage)
    read_time_percentages.append(read_percentage)
    write_time_percentages.append(write_percentage)

# Calculate deviation percentages for computation and I/O times
computation_deviation = calculate_deviation_percentage(computation_times)
io_deviation = calculate_deviation_percentage(io_times)

# Calculate averaged I/O and computation time percentages
averaged_io_time_percentages, averaged_computation_time_percentages = calculate_averaged_percentages(io_percentages)

# Calculate averaged task time percentages, read percentages, and write percentages
averaged_task_time_percentages = {
    task: np.mean([trial[task] for trial in task_time_percentages])
    for task in task_time_percentages[0].keys()
}

averaged_read_time_percentages = {
    task: np.mean([trial[task] for trial in read_time_percentages])
    for task in read_time_percentages[0].keys()
}

averaged_write_time_percentages = {
    task: np.mean([trial[task] for trial in write_time_percentages])
    for task in write_time_percentages[0].keys()
}

# Display Results
print("\nDeviation Percentages of Computation Times (by task):")
for task, deviation in computation_deviation.items():
    print(f"Task {task}: {deviation:.2f}%")

print("\nDeviation Percentages of I/O Times (by task):")
for task, deviation in io_deviation.items():
    print(f"Task {task}: {deviation:.2f}%")

print("\nAveraged I/O Time Percentages (by task):")
for task, average in averaged_io_time_percentages.items():
    print(f"Task {task}: {average:.2f}%")

print("\nAveraged Computation Time Percentages (by task):")
for task, average in averaged_computation_time_percentages.items():
    print(f"Task {task}: {average:.2f}%")

print("\nAveraged Task Time Percentages (by task):")
for task, average in averaged_task_time_percentages.items():
    print(f"Task {task}: {average:.2f}%")

print("\nAveraged Read Time Percentages (by task):")
for task, average in averaged_read_time_percentages.items():
    print(f"Task {task}: {average:.2f}%")

print("\nAveraged Write Time Percentages (by task):")
for task, average in averaged_write_time_percentages.items():
    print(f"Task {task}: {average:.2f}%")



Deviation Percentages of Computation Times (by task):
Task individuals: 55.33%
Task individuals_merge: 45.39%
Task sifting: 57.91%
Task mutation_overlap: 11.67%
Task frequency: 10.20%

Deviation Percentages of I/O Times (by task):
Task individuals: 2.88%
Task individuals_merge: 0.46%
Task sifting: 0.59%
Task mutation_overlap: 33.58%
Task frequency: 123.98%

Averaged I/O Time Percentages (by task):
Task individuals: 80.37%
Task individuals_merge: 0.44%
Task sifting: 3.62%
Task mutation_overlap: 0.82%
Task frequency: 0.13%

Averaged Computation Time Percentages (by task):
Task individuals: 19.63%
Task individuals_merge: 99.56%
Task sifting: 96.38%
Task mutation_overlap: 99.18%
Task frequency: 99.87%

Averaged Task Time Percentages (by task):
Task individuals: 16.28%
Task individuals_merge: 12.91%
Task sifting: 3.33%
Task mutation_overlap: 3.35%
Task frequency: 64.12%

Averaged Read Time Percentages (by task):
Task individuals: 99.99%
Task individuals_merge: 99.52%
Task sifting: 99.75%
T